# 03 — Train the foundation model

Self-supervised pretraining (masked reconstruction + next-day + contrastive)
followed by multi-task downstream fine-tuning.

For the full pipeline use the CLI script:

```bash
python scripts/train_model.py
```

This notebook runs a small version end-to-end for readability.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import torch
import matplotlib.pyplot as plt

from lhfm.data.synthetic_generator import generate_synthetic_cohort
from lhfm.features import build_full_feature_table
from lhfm.data.preprocessing import build_windows, train_val_test_split_by_participant
from lhfm.models.encoder import MultimodalLongitudinalEncoder
from lhfm.models.self_supervised import SSLLossWeights
from lhfm.training.dataset import LongitudinalWindowDataset
from lhfm.training.train_ssl import pretrain_ssl
from lhfm.training.train_downstream import train_downstream
from lhfm.utils.config import set_global_seed
set_global_seed(42)

In [ ]:
raw = generate_synthetic_cohort(n_participants=120, n_days=75, seed=42)
feat = build_full_feature_table(raw, impute=True, add_targets=True)

FEATURE_GROUPS = {
    'wearable':   ['sleep_duration','sleep_efficiency','sleep_regularity_index',
                   'hrv_dev_from_baseline','rhr_dev_from_baseline','stress_burden_7d'],
    'smartphone': ['screen_time_z','unlock_freq_z','mobility_radius_km','location_entropy','behavioral_regularity'],
    'climate':    ['temperature_c','heat_index','aqi','humid_heat_index','nighttime_heat_stress'],
    'baseline':   ['age_z','chronotype_score','baseline_hrv','baseline_sleep_need'],
}
feature_cols = sum(FEATURE_GROUPS.values(), [])
modality_dims = {k: len(v) for k, v in FEATURE_GROUPS.items()}

modality_slices, cursor = {}, 0
for k, cols in FEATURE_GROUPS.items():
    modality_slices[k] = (cursor, cursor + len(cols)); cursor += len(cols)

TASK_TO_COL = {'low_mood':'target_low_mood','high_stress':'target_high_stress',
               'sleep_disruption':'target_sleep_disruption','climate_vulnerable':'target_climate_vulnerable'}
task_names = list(TASK_TO_COL.keys())

In [ ]:
splits = train_val_test_split_by_participant(feat, val_fraction=0.15, test_fraction=0.15, seed=42)

def build_split(split_df):
    X, _, pids, _ = build_windows(split_df, feature_cols=feature_cols,
                                  target_col=TASK_TO_COL[task_names[0]],
                                  window_days=14, stride=1, target_mode='next_day')
    Y = np.full((len(X), len(task_names)), np.nan, dtype=np.float32)
    for i, t in enumerate(task_names):
        _, y, _, _ = build_windows(split_df, feature_cols=feature_cols,
                                   target_col=TASK_TO_COL[t], window_days=14,
                                   stride=1, target_mode='next_day')
        Y[:, i] = y
    return X, Y, pids

Xtr, Ytr, pids_tr = build_split(splits['train'])
Xva, Yva, pids_va = build_split(splits['val'])
Xte, Yte, pids_te = build_split(splits['test'])
print('windows:', Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
train_ds = LongitudinalWindowDataset(Xtr, Ytr, modality_slices)
val_ds   = LongitudinalWindowDataset(Xva, Yva, modality_slices)
test_ds  = LongitudinalWindowDataset(Xte, Yte, modality_slices)

encoder = MultimodalLongitudinalEncoder(
    modality_dims=modality_dims,
    d_model=64, n_heads=4, n_layers=2, max_seq_len=14,
    n_participants=0,
)
encoder

### SSL pretraining

In [ ]:
ssl_state = pretrain_ssl(
    encoder, train_dataset=train_ds, val_dataset=val_ds,
    epochs=8, batch_size=32, lr=5e-4,
    mask_ratio=0.20,
    weights=SSLLossWeights(recon=1.0, next_day=0.5, contrastive=0.25, temperature=0.1),
    device='cpu', early_stopping_patience=5,
)
plt.figure(figsize=(6,3))
plt.plot(ssl_state.train_losses, label='train')
plt.plot(ssl_state.val_losses, label='val')
plt.xlabel('epoch'); plt.ylabel('ssl loss'); plt.legend(); plt.grid(alpha=0.3); plt.show()

### Downstream fine-tuning

In [ ]:
ds_state = train_downstream(
    encoder=encoder, task_names=task_names,
    train_dataset=train_ds, val_dataset=val_ds,
    epochs=15, batch_size=32, lr=5e-4, device='cpu',
    freeze_encoder=False, early_stopping_patience=5,
)
plt.figure(figsize=(6,3))
plt.plot(ds_state.train_losses, label='train')
plt.plot(ds_state.val_losses, label='val')
plt.xlabel('epoch'); plt.ylabel('downstream loss'); plt.legend(); plt.grid(alpha=0.3); plt.show()